# ⚙️ CORE PIPELINE: HỆ THỐNG SỐ HÓA TỦ SÁCH
**Thực hiện:** Nguyễn Tùng Lâm

Notebook trình bày quy trình tích hợp giữa Tiền xử lý, Nhận diện YOLOv5x6 và VietOCR.

## A. Thiết lập Môi trường

In [ ]:
# 1. Clone và Cài đặt
import os, shutil
%cd /content/
if os.path.exists('bookcase-digitization'): shutil.rmtree('bookcase-digitization')
!git clone https://github.com/pie-12/bookcase-digitization.git
%cd bookcase-digitization

print("🛠 Đang cài đặt thư viện lõi...")
!pip install "numpy<2" opencv-python-headless==4.8.0.74 --force-reinstall -q
!pip install craft-text-detector vietocr==0.3.5 --no-deps -q
!pip install albumentations==1.4.2 einops gdown prefetch-generator shapely scikit-image -q
!pip install -q ultralytics
!git clone https://github.com/ultralytics/yolov5 -q

# Vá lỗi mã nguồn CRAFT/VietOCR ngay tại chỗ
vgg_path = "/usr/local/lib/python3.12/dist-packages/craft_text_detector/models/basenet/vgg16_bn.py"
if os.path.exists(vgg_path):
    with open(vgg_path, 'r') as f: content = f.read()
    with open(vgg_path, 'w') as f: f.write(content.replace("from torchvision.models.vgg import model_urls", "model_urls = {}"))

print("✅ Môi trường đã sẵn sàng.")

## B. Nạp mô hình Huấn luyện (Weights)

In [ ]:
from google.colab import files
import os

print("HÃY CHỌN FILE best.pt TỪ MÁY TÍNH CỦA BẠN:")
uploaded = files.upload()

for fn in uploaded.keys():
    if fn == 'best.pt':
        if os.path.exists('last.pt'): os.remove('last.pt')
        os.rename('best.pt', 'last.pt')
        print("✅ Đã nạp mô hình: last.pt")

## C. Lựa chọn Dữ liệu Hình ảnh

In [ ]:
import os
test_images = sorted([f for f in os.listdir('data_test') if f.lower().endswith(('.jpg', '.png'))])

print("Danh sách ảnh có sẵn trong hệ thống:")
for i, name in enumerate(test_images):
    print(f"{i+1}. {name}")

print("\n--- 🚀 HỆ THỐNG SẼ CHẠY NHẬN DIỆN CHO TOÀN BỘ DANH SÁCH TRÊN ---")

## D. Thực thi Pipeline & Trực quan hóa

In [ ]:
# Chạy trích xuất chính thức
!python run_inference.py

# Hiển thị một kết quả ngẫu nhiên để minh họa
from IPython.display import Image, display
import random

sample_img = "detected_" + random.choice(test_images)
img_path = os.path.join('runs/detect', sample_img)

if os.path.exists(img_path):
    print(f"\n🖼️ Ảnh minh họa kết quả: {sample_img}")
    display(Image(filename=img_path, width=500))

import pandas as pd
if os.path.exists('final_results.csv'):
    print("\n✅ Bảng dữ liệu trích xuất thành công:")
    df = pd.read_csv('final_results.csv')
    display(df.head(10))